In [ ]:
# CELL 1 — Imports

import os
import time
import gc
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models

print("[OK] All imports successful")
print(f"PyTorch version: {torch.__version__}")


In [ ]:
KAGGLE_INPUT = "/kaggle/input"
dataset_folders = os.listdir(KAGGLE_INPUT)
DATASET_PATH = None
for folder in dataset_folders:
    candidate = os.path.join(KAGGLE_INPUT, folder)
    if os.path.isdir(os.path.join(candidate, "train_Low_Resolution")):
        DATASET_PATH = candidate
        break
assert DATASET_PATH is not None, f"Dataset not found in {KAGGLE_INPUT}. Folders: {dataset_folders}"
print(f"[INFO] Dataset found at: {DATASET_PATH}")

TRAIN_LR = os.path.join(DATASET_PATH, "train_Low_Resolution")
TRAIN_HR = os.path.join(DATASET_PATH, "train_High_Resolution")
TEST_LR  = os.path.join(DATASET_PATH, "test_Low_Resolution")
VGG_PATH = os.path.join(DATASET_PATH, "vgg19_weights.pth")

for p in [TRAIN_LR, TRAIN_HR, TEST_LR, VGG_PATH]:
    assert os.path.exists(p), f"MISSING PATH: {p}"
    print(f"[OK] {p}")

print(f"\nTrain LR images: {len(os.listdir(TRAIN_LR))}")
print(f"Train HR images: {len(os.listdir(TRAIN_HR))}")
print(f"Test  LR images: {len(os.listdir(TEST_LR))}")


In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f"[GPU] {torch.cuda.get_device_name(0)}")
    print(f"[VRAM] {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("[CPU] No GPU found — running on CPU")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_worker(worker_id):
    np.random.seed(SEED + worker_id)
    random.seed(SEED + worker_id)

g = torch.Generator()
g.manual_seed(SEED)

print(f"[OK] Device: {device}, Seed: {SEED}")


In [ ]:
# Hyperparameter config block

# ── Epochs & Training ──
WARMUP_EPOCHS    = 30
GAN_EPOCHS       = 70
TOTAL_EPOCHS     = WARMUP_EPOCHS + GAN_EPOCHS

BATCH_SIZE       = 8
GRAD_ACCUM_STEPS = 2
LR_G             = 1e-4
LR_D             = 1e-4
BETAS            = (0.9, 0.999)

# ── Loss Weights ──
LAMBDA_PIXEL      = 1.0
LAMBDA_PERCEPTUAL = 0.006
LAMBDA_ADV        = 0.001

# ── Early Stopping ──
PATIENCE          = 15
RUNTIME_LIMIT_HRS = 8.0

# ── Image Dimensions ──
LR_SIZE  = 32
HR_SIZE  = 128
CHANNELS = 3

# ── Paths ──
CHECKPOINT_PATH = "/kaggle/working/best_generator.pth"
SUBMISSION_PATH = "/kaggle/working/submission.csv"

# ── Mixed Precision ──
USE_AMP = torch.cuda.is_available()

print(f"[CONFIG] Warmup={WARMUP_EPOCHS}, GAN={GAN_EPOCHS}, Total={TOTAL_EPOCHS}")
print(f"[CONFIG] Batch={BATCH_SIZE}, GradAccum={GRAD_ACCUM_STEPS}, EffBatch={BATCH_SIZE*GRAD_ACCUM_STEPS}")
print(f"[CONFIG] LR_G={LR_G}, LR_D={LR_D}, AMP={USE_AMP}")
print(f"[CONFIG] Patience={PATIENCE}, RuntimeLimit={RUNTIME_LIMIT_HRS}h")


In [ ]:
# Dataset class + train/val split + DataLoaders

class PlantLeafSRDataset(Dataset):
    """Paired LR/HR dataset for plant leaf super-resolution."""
    def __init__(self, lr_dir, hr_dir, augment=False):
        super().__init__()
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.augment = augment
        self.filenames = sorted([
            f for f in os.listdir(lr_dir)
            if f.lower().endswith(('.png', '.jpg', '.jpeg'))
        ])
        print(f"[Dataset] Loaded {len(self.filenames)} image pairs, augment={augment}")

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        lr_path = os.path.join(self.lr_dir, fname)
        hr_path = os.path.join(self.hr_dir, fname)

        lr_img = Image.open(lr_path).convert("RGB")
        hr_img = Image.open(hr_path).convert("RGB")

        lr_np = np.array(lr_img, dtype=np.float32)
        hr_np = np.array(hr_img, dtype=np.float32)

        if self.augment:
            if random.random() > 0.5:
                lr_np = np.flip(lr_np, axis=1).copy()
                hr_np = np.flip(hr_np, axis=1).copy()
            if random.random() > 0.5:
                lr_np = np.flip(lr_np, axis=0).copy()
                hr_np = np.flip(hr_np, axis=0).copy()

        lr_tensor = torch.from_numpy(lr_np / 127.5 - 1.0).permute(2, 0, 1).float()
        hr_tensor = torch.from_numpy(hr_np / 127.5 - 1.0).permute(2, 0, 1).float()

        return lr_tensor, hr_tensor

full_dataset = PlantLeafSRDataset(TRAIN_LR, TRAIN_HR, augment=True)
total_size = len(full_dataset)
train_size = int(0.9 * total_size)
val_size   = total_size - train_size

split_gen = torch.Generator().manual_seed(SEED)
train_dataset, val_dataset_raw = random_split(full_dataset, [train_size, val_size], generator=split_gen)

class ValWrapper(Dataset):
    """Wraps a Subset to disable augmentation."""
    def __init__(self, subset, lr_dir, hr_dir):
        self.subset = subset
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.filenames = [full_dataset.filenames[i] for i in subset.indices]

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        lr_path = os.path.join(self.lr_dir, fname)
        hr_path = os.path.join(self.hr_dir, fname)
        lr_img = Image.open(lr_path).convert("RGB")
        hr_img = Image.open(hr_path).convert("RGB")
        lr_np = np.array(lr_img, dtype=np.float32)
        hr_np = np.array(hr_img, dtype=np.float32)
        lr_tensor = torch.from_numpy(lr_np / 127.5 - 1.0).permute(2, 0, 1).float()
        hr_tensor = torch.from_numpy(hr_np / 127.5 - 1.0).permute(2, 0, 1).float()
        return lr_tensor, hr_tensor

val_dataset = ValWrapper(val_dataset_raw, TRAIN_LR, TRAIN_HR)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
    worker_init_fn=seed_worker, generator=g
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
    worker_init_fn=seed_worker, generator=g
)

print(f"[OK] Train: {len(train_dataset)}, Val: {len(val_dataset)}")
print(f"[OK] Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")


In [ ]:
# DenseBlock + RRDB building blocks

class DenseBlock(nn.Module):
    """5-layer dense block with growth rate 32. No BatchNorm."""
    def __init__(self, in_channels=64, growth_rate=32):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels,                    growth_rate, 3, padding=1)
        self.conv2 = nn.Conv2d(in_channels + growth_rate * 1,  growth_rate, 3, padding=1)
        self.conv3 = nn.Conv2d(in_channels + growth_rate * 2,  growth_rate, 3, padding=1)
        self.conv4 = nn.Conv2d(in_channels + growth_rate * 3,  growth_rate, 3, padding=1)
        self.conv5 = nn.Conv2d(in_channels + growth_rate * 4,  in_channels, 3, padding=1)
        self.lrelu = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        x1 = self.lrelu(self.conv1(x))
        x2 = self.lrelu(self.conv2(torch.cat([x, x1], 1)))
        x3 = self.lrelu(self.conv3(torch.cat([x, x1, x2], 1)))
        x4 = self.lrelu(self.conv4(torch.cat([x, x1, x2, x3], 1)))
        x5 = self.conv5(torch.cat([x, x1, x2, x3, x4], 1))
        return x5 * 0.2 + x


class RRDB(nn.Module):
    """Residual-in-Residual Dense Block: 3 DenseBlocks with residual scaling."""
    def __init__(self, in_channels=64, growth_rate=32):
        super().__init__()
        self.db1 = DenseBlock(in_channels, growth_rate)
        self.db2 = DenseBlock(in_channels, growth_rate)
        self.db3 = DenseBlock(in_channels, growth_rate)

    def forward(self, x):
        out = self.db1(x)
        out = self.db2(out)
        out = self.db3(out)
        return out * 0.2 + x

print("[OK] DenseBlock and RRDB defined")


In [ ]:
# Generator definition + weight init

class Generator(nn.Module):
    """ESRGAN-inspired RRDB-Net generator for 4x super-resolution."""
    def __init__(self, in_channels=3, out_channels=3, num_features=64, num_rrdb=16):
        super().__init__()
        self.conv_first = nn.Conv2d(in_channels, num_features, 3, padding=1)
        self.trunk = nn.Sequential(*[RRDB(num_features, 32) for _ in range(num_rrdb)])
        self.conv_last_trunk = nn.Conv2d(num_features, num_features, 3, padding=1)

        self.up1_conv = nn.Conv2d(num_features, num_features * 4, 3, padding=1)
        self.up1_ps   = nn.PixelShuffle(2)
        self.up2_conv = nn.Conv2d(num_features, num_features * 4, 3, padding=1)
        self.up2_ps   = nn.PixelShuffle(2)

        self.hr_conv  = nn.Conv2d(num_features, num_features, 3, padding=1)
        self.conv_last = nn.Conv2d(num_features, out_channels, 3, padding=1)

        # Separate skip branch layers (avoid reusing main branch with inplace ops)
        self.skip_hr_conv  = nn.Conv2d(num_features, num_features, 3, padding=1)
        self.skip_conv_last = nn.Conv2d(num_features, out_channels, 3, padding=1)

        self.lrelu = nn.LeakyReLU(0.2, inplace=True)
        self.tanh  = nn.Tanh()

    def forward(self, x):
        feat_first = self.conv_first(x)
        trunk_out  = self.conv_last_trunk(self.trunk(feat_first))
        feat = feat_first + trunk_out

        feat = self.lrelu(self.up1_ps(self.up1_conv(feat)))
        feat = self.lrelu(self.up2_ps(self.up2_conv(feat)))
        feat = self.lrelu(self.hr_conv(feat))
        out  = self.tanh(self.conv_last(feat))

        skip = F.interpolate(feat_first, scale_factor=4, mode='bilinear', align_corners=False)
        skip = self.tanh(self.skip_conv_last(self.lrelu(self.skip_hr_conv(skip))))

        return torch.clamp(out + skip, -1.0, 1.0)


def init_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, mode='fan_out', nonlinearity='leaky_relu')
        if module.bias is not None:
            nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.BatchNorm2d):
        nn.init.constant_(module.weight, 1)
        nn.init.constant_(module.bias, 0)
    elif isinstance(module, nn.Linear):
        nn.init.normal_(module.weight, 0, 0.01)
        nn.init.constant_(module.bias, 0)

generator = Generator().to(device)
generator.apply(init_weights)

total_params_g = sum(p.numel() for p in generator.parameters())
print(f"[OK] Generator created: {total_params_g / 1e6:.2f}M parameters")


In [ ]:
# Discriminator definition + weight init

class Discriminator(nn.Module):
    """VGG-style PatchGAN discriminator for 128x128 HR images."""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3,   64,  3, stride=1, padding=1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64,  64,  4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64,  128, 3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 128, 4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 256, 4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 3, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
        )
        self.classifier = nn.Linear(512, 1)

    def forward(self, x):
        feat = self.features(x)
        return self.classifier(feat)

discriminator = Discriminator().to(device)
discriminator.apply(init_weights)

total_params_d = sum(p.numel() for p in discriminator.parameters())
print(f"[OK] Discriminator created: {total_params_d / 1e6:.2f}M parameters")


In [ ]:
# VGGPerceptualLoss class (loads from VGG_PATH, frozen)

class VGGPerceptualLoss(nn.Module):
    """Perceptual loss using VGG19 features loaded from offline weights file."""
    def __init__(self, vgg_path):
        super().__init__()
        vgg = models.vgg19(pretrained=False)
        vgg.load_state_dict(torch.load(vgg_path, map_location='cpu'))
        self.feature_extractor = nn.Sequential(*list(vgg.features)[:18])
        for param in self.feature_extractor.parameters():
            param.requires_grad = False
        self.feature_extractor.eval()

    def forward(self, generated, target):
        gen_vgg = (generated + 1.0) / 2.0
        tgt_vgg = (target + 1.0) / 2.0
        gen_feat = self.feature_extractor(gen_vgg)
        tgt_feat = self.feature_extractor(tgt_vgg)
        return F.mse_loss(gen_feat, tgt_feat)

perceptual_loss_fn = VGGPerceptualLoss(VGG_PATH).to(device)
print("[OK] VGGPerceptualLoss loaded and frozen")


In [ ]:
# CELL 10 — Loss functions + Optimizers + Schedulers + AMP scalers

criterion_pixel = nn.L1Loss().to(device)
criterion_adv   = nn.BCEWithLogitsLoss().to(device)

optimizer_G = optim.Adam(generator.parameters(),     lr=LR_G, betas=BETAS)
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR_D, betas=BETAS)

scheduler_G = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_G, mode='min', factor=0.5, patience=10, verbose=True
)

scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("[OK] Losses, optimizers, scheduler, and AMP scalers initialized")


In [ ]:
# Phase 1: Warmup training loop (Generator only, pixel loss)

def validate_mae(gen, loader):
    """Compute validation MAE in pixel space [0,255]."""
    gen.eval()
    total_mae = 0.0
    count = 0
    with torch.no_grad():
        for lr_batch, hr_batch in loader:
            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                sr_batch = gen(lr_batch)
            sr_255 = (sr_batch.float() + 1.0) * 127.5
            hr_255 = (hr_batch.float() + 1.0) * 127.5
            total_mae += F.l1_loss(sr_255, hr_255, reduction='sum').item()
            count += hr_batch.numel()
    gen.train()
    return total_mae / count

print("=" * 60)
print("PHASE 1: Generator Warmup (pixel loss only)")
print("=" * 60)

best_val_mae = float('inf')
patience_count = 0
train_start = time.time()

try:
    for epoch in range(1, WARMUP_EPOCHS + 1):
        elapsed_hrs = (time.time() - train_start) / 3600
        if elapsed_hrs > RUNTIME_LIMIT_HRS:
            print(f"[TIMEOUT GUARD] {elapsed_hrs:.2f}h elapsed — stopping training early.")
            break

        generator.train()
        epoch_g_loss = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f"Warmup Epoch {epoch}/{WARMUP_EPOCHS}")
        for batch_idx, (lr_batch, hr_batch) in enumerate(pbar):
            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                sr_batch = generator(lr_batch)
                loss_pixel = criterion_pixel(sr_batch, hr_batch)
                loss_G = (LAMBDA_PIXEL * loss_pixel) / GRAD_ACCUM_STEPS

            scaler_G.scale(loss_G).backward()

            if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                scaler_G.step(optimizer_G)
                scaler_G.update()
                optimizer_G.zero_grad(set_to_none=True)

            epoch_g_loss += loss_pixel.item()
            num_batches += 1
            pbar.set_postfix({"pixel_loss": f"{loss_pixel.item():.4f}"})

        avg_g_loss = epoch_g_loss / max(num_batches, 1)
        val_mae = validate_mae(generator, val_loader)
        scheduler_G.step(val_mae)

        elapsed_hrs = (time.time() - train_start) / 3600
        print(f"Epoch [{epoch}/{TOTAL_EPOCHS}] "
              f"G_loss: {avg_g_loss:.4f} | D_loss: N/A    | "
              f"Val_MAE: {val_mae:.4f} | LR: {optimizer_G.param_groups[0]['lr']:.2e} | "
              f"Time: {elapsed_hrs:.2f}h")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_count = 0
            torch.save(generator.state_dict(), CHECKPOINT_PATH)
            print(f"[SAVED] Best model at epoch {epoch}, MAE={val_mae:.4f}")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"[EARLY STOP] No improvement for {PATIENCE} epochs.")
                break

except Exception as e:
    print(f"[ERROR] Phase 1 crashed: {e}")
    if os.path.exists(CHECKPOINT_PATH):
        print(f"[INFO] Checkpoint exists at {CHECKPOINT_PATH}")
    else:
        torch.save(generator.state_dict(), CHECKPOINT_PATH)
        print(f"[EMERGENCY SAVE] Saved checkpoint after crash")

print(f"\n[Phase 1 Complete] Best Val MAE: {best_val_mae:.4f}")


In [ ]:
# Memory cleanup between phases

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()
print("[OK] Memory cleaned between Phase 1 and Phase 2")


In [ ]:
# Full GAN training loop (combined loss)

print("=" * 60)
print("PHASE 2: Full cGAN Training")
print("=" * 60)

generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
generator.train()
discriminator.train()

optimizer_G = optim.Adam(generator.parameters(),     lr=LR_G, betas=BETAS)
optimizer_D = optim.Adam(discriminator.parameters(), lr=LR_D, betas=BETAS)
scheduler_G = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_G, mode='min', factor=0.5, patience=10, verbose=True
)
scaler_G = torch.cuda.amp.GradScaler(enabled=USE_AMP)
scaler_D = torch.cuda.amp.GradScaler(enabled=USE_AMP)

patience_count = 0
phase2_start = time.time()

try:
    for epoch in range(WARMUP_EPOCHS + 1, TOTAL_EPOCHS + 1):
        elapsed_total_hrs = (time.time() - train_start) / 3600
        if elapsed_total_hrs > RUNTIME_LIMIT_HRS:
            print(f"[TIMEOUT GUARD] {elapsed_total_hrs:.2f}h elapsed — stopping training early.")
            break

        generator.train()
        discriminator.train()
        epoch_g_loss = 0.0
        epoch_d_loss = 0.0
        num_batches = 0

        pbar = tqdm(train_loader, desc=f"GAN Epoch {epoch}/{TOTAL_EPOCHS}")
        for batch_idx, (lr_batch, hr_batch) in enumerate(pbar):
            lr_batch = lr_batch.to(device, non_blocking=True)
            hr_batch = hr_batch.to(device, non_blocking=True)
            batch_size_actual = lr_batch.size(0)

            real_labels = torch.ones(batch_size_actual, 1, device=device)
            fake_labels = torch.zeros(batch_size_actual, 1, device=device)

            # ── Train Discriminator ──
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                sr_batch = generator(lr_batch).detach()
                pred_real = discriminator(hr_batch)
                pred_fake = discriminator(sr_batch)
                loss_d_real = criterion_adv(pred_real, real_labels)
                loss_d_fake = criterion_adv(pred_fake, fake_labels)
                loss_D = (loss_d_real + loss_d_fake) / 2.0 / GRAD_ACCUM_STEPS

            scaler_D.scale(loss_D).backward()

            if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                scaler_D.step(optimizer_D)
                scaler_D.update()
                optimizer_D.zero_grad(set_to_none=True)

            # ── Train Generator ──
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                sr_batch = generator(lr_batch)
                pred_fake_for_g = discriminator(sr_batch)
                loss_pixel = criterion_pixel(sr_batch, hr_batch)
                loss_perc  = perceptual_loss_fn(sr_batch, hr_batch)
                loss_adv   = criterion_adv(pred_fake_for_g, real_labels)
                loss_G = (LAMBDA_PIXEL * loss_pixel +
                          LAMBDA_PERCEPTUAL * loss_perc +
                          LAMBDA_ADV * loss_adv) / GRAD_ACCUM_STEPS

            scaler_G.scale(loss_G).backward()

            if (batch_idx + 1) % GRAD_ACCUM_STEPS == 0 or (batch_idx + 1) == len(train_loader):
                scaler_G.step(optimizer_G)
                scaler_G.update()
                optimizer_G.zero_grad(set_to_none=True)

            epoch_g_loss += (LAMBDA_PIXEL * loss_pixel + LAMBDA_PERCEPTUAL * loss_perc + LAMBDA_ADV * loss_adv).item()
            epoch_d_loss += ((loss_d_real + loss_d_fake) / 2.0).item()
            num_batches += 1
            pbar.set_postfix({
                "G": f"{loss_pixel.item():.4f}",
                "D": f"{((loss_d_real + loss_d_fake) / 2.0).item():.4f}"
            })

        avg_g_loss = epoch_g_loss / max(num_batches, 1)
        avg_d_loss = epoch_d_loss / max(num_batches, 1)
        val_mae = validate_mae(generator, val_loader)
        scheduler_G.step(val_mae)

        elapsed_hrs = (time.time() - train_start) / 3600
        print(f"Epoch [{epoch}/{TOTAL_EPOCHS}] "
              f"G_loss: {avg_g_loss:.4f} | D_loss: {avg_d_loss:.4f} | "
              f"Val_MAE: {val_mae:.4f} | LR: {optimizer_G.param_groups[0]['lr']:.2e} | "
              f"Time: {elapsed_hrs:.2f}h")

        if val_mae < best_val_mae:
            best_val_mae = val_mae
            patience_count = 0
            torch.save(generator.state_dict(), CHECKPOINT_PATH)
            print(f"[SAVED] Best model at epoch {epoch}, MAE={val_mae:.4f}")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"[EARLY STOP] No improvement for {PATIENCE} epochs.")
                break

except Exception as e:
    print(f"[ERROR] Phase 2 crashed: {e}")
    if os.path.exists(CHECKPOINT_PATH):
        print(f"[INFO] Checkpoint exists at {CHECKPOINT_PATH}")
    else:
        torch.save(generator.state_dict(), CHECKPOINT_PATH)
        print(f"[EMERGENCY SAVE] Saved checkpoint after crash")

print(f"\n[Phase 2 Complete] Best Val MAE: {best_val_mae:.4f}")
print(f"Total training time: {(time.time() - train_start) / 3600:.2f}h")


In [ ]:
# Inference loop (load best checkpoint, generate all 495 HR images)

print("=" * 60)
print("INFERENCE: Generating super-resolved images")
print("=" * 60)

generator.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
generator.eval()
print(f"[OK] Loaded best checkpoint from {CHECKPOINT_PATH}")

test_files = sorted(os.listdir(TEST_LR))
test_files = [f for f in test_files if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
print(f"[INFO] Found {len(test_files)} test images")

rows = []
with torch.no_grad():
    for fname in tqdm(test_files, desc="Generating HR images"):
        img_path = os.path.join(TEST_LR, fname)
        img = Image.open(img_path).convert("RGB")

        lr_tensor = torch.tensor(
            np.array(img, dtype=np.float32) / 127.5 - 1.0
        ).permute(2, 0, 1).unsqueeze(0).to(device)

        with torch.cuda.amp.autocast(enabled=USE_AMP):
            sr_tensor = generator(lr_tensor)

        sr_np = sr_tensor.squeeze(0).cpu().float().numpy()
        sr_np = (sr_np + 1.0) * 127.5
        sr_np = np.clip(sr_np, 0, 255).astype(np.uint8)

        sr_hwc = sr_np.transpose(1, 2, 0)
        flat = sr_hwc.flatten()

        assert flat.shape[0] == 49152, \
            f"[ERROR] {fname}: expected 49152 values, got {flat.shape[0]}"
        assert flat.min() >= 0 and flat.max() <= 255, \
            f"[ERROR] {fname}: pixel values out of [0,255] range"
        assert flat.dtype == np.uint8, \
            f"[ERROR] {fname}: dtype must be uint8, got {flat.dtype}"

        pixel_str = " ".join(map(str, flat.tolist()))
        rows.append({"Id": fname, "Pixels": pixel_str})

print(f"[OK] Generated {len(rows)} super-resolved images")


In [ ]:
#  CSV generation with full validation

print("=" * 60)
print("CSV GENERATION & VALIDATION")
print("=" * 60)

submission_df = pd.DataFrame(rows, columns=["Id", "Pixels"])

assert len(submission_df) == 495, \
    f"[ERROR] Expected 495 rows, got {len(submission_df)}"
assert list(submission_df.columns) == ["Id", "Pixels"], \
    "[ERROR] Column names must be exactly ['Id', 'Pixels']"

sample_pixels = submission_df["Pixels"].iloc[0].split(" ")
assert len(sample_pixels) == 49152, \
    f"[ERROR] First row has {len(sample_pixels)} values, expected 49152"
assert all(v.isdigit() for v in sample_pixels[:10]), \
    "[ERROR] Pixel values must be plain integers"

print(f"[OK] submission.csv validated: {len(submission_df)} rows, 49152 values each")

submission_df.to_csv(SUBMISSION_PATH, index=False)
print(f"[SAVED] {SUBMISSION_PATH}")
print(submission_df.head(3).to_string())


In [ ]:
# Final sanity check

print("=" * 60)
print("FINAL SANITY CHECK")
print("=" * 60)

final_df = pd.read_csv(SUBMISSION_PATH)
print(f"Shape: {final_df.shape}")
print(f"Columns: {list(final_df.columns)}")
print(f"\nHead(3):")
print(final_df.head(3).to_string())

for i in range(min(3, len(final_df))):
    row_pixels = final_df["Pixels"].iloc[i].split(" ")
    print(f"  Row {i} ({final_df['Id'].iloc[i]}): {len(row_pixels)} values, "
          f"range [{min(int(v) for v in row_pixels)}-{max(int(v) for v in row_pixels)}]")

file_size_mb = os.path.getsize(SUBMISSION_PATH) / (1024 * 1024)
print(f"\n[FINAL] submission.csv size: {file_size_mb:.2f} MB")
print(f"[FINAL] Total rows: {len(final_df)}")
print(f"[FINAL] Total training time: {(time.time() - train_start) / 3600:.2f}h")
print("[DONE] Notebook execution complete!")
